# Interactive Exploration: `GDALocations` API Tour

This notebook explores the `GDALocations` master jurisdiction manager:
1. Dynamic workspace root path anchoring.
2. Loading canonical test fixtures vs. production registries.
3. Offline US state normalization via the `us` package.
4. Exact, wildcard, and fuzzy queries across canonical names and aliases.
5. Hierarchical queries (`find_by_jurisdiction`).
6. Temporal validity and boundary lineage tracking (predecessor / successor splits).
7. Alias mutation and isolated persistence testing targeting `gtemp/`.

In [1]:
from pathlib import Path
import os
import sys

# Ensure kernel working directory is anchored to the archive root
cwd = Path.cwd().resolve()
if cwd.name in ("exploration", "maintenance", "notebooks"):
    repo_root = cwd.parents[1] if cwd.name in ("exploration", "maintenance") else cwd.parent
    os.chdir(repo_root)
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Workspace Root: {Path.cwd()}")

Workspace Root: G:\My Drive\genealogy-digital-archive


In [2]:
from tools.lib.gda_core.GDALocations import GDALocations
from tools.lib.gda_core.GDAConfig import CONFIG

# Load from golden fixture to keep production untouched
golden_fixture = Path("tests/fixtures/locations/golden/golden_locations.json")
if not golden_fixture.exists():
    golden_fixture = Path("data/entities/locations.json")

locations = GDALocations(filepath=golden_fixture, auto_load=True)

print(f"Loaded Locations Count: {locations.count}")
print(f"Registry Dirty State:    {locations.is_dirty}")

Loaded Locations Count: 105
Registry Dirty State:    False


In [3]:
# 1. US State Normalization Helper (Offline via 'us' package)
sample_abbrs = ["PA", "nh", "Penn", "texas", "Bavaria"]
for abbr in sample_abbrs:
    norm = GDALocations.normalize_state("USA", abbr)
    print(f"{abbr!r:>10} -> {norm}")

      'PA' -> Pennsylvania
      'nh' -> New Hampshire
    'Penn' -> Penn
   'texas' -> Texas
 'Bavaria' -> Bavaria


In [ ]:
# 2. Primary Key Lookup & Record Inspection
loc = locations.get_by_id("LOC-00006")
loc

In [ ]:
# 3. Hierarchical Jurisdiction Lookup
match = locations.find_by_jurisdiction(
    country="USA",
    state_or_province="PA",
    county="Cumberland",
    local_jurisdiction="Carlisle"
)
if match:
    print(f"Found: [{match['location_id']}] {match['canonical_name']}")
else:
    print("Jurisdiction not found.")

In [ ]:
# 4. Search: Wildcard & Fuzzy Resolution
print("--- Wildcard Search (*Frankford*) ---")
for m in locations.search(query="*Frankford*", use_fuzzy=False):
    print(f"[{m['location_id']}] {m['canonical_name']}")

print("\n--- Fuzzy Search ('Hbg' with typos) ---")
for m in locations.search(query="Harisburg", use_fuzzy=True, fuzzy_threshold=0.8):
    print(f"[{m['location_id']}] {m['canonical_name']} (Aliases: {m.get('aliases')})")

In [ ]:
# 5. Temporal Validity & Boundary Lineage Checks
# Checking if a jurisdiction was extant at specific dates
target_loc = "LOC-00015"
record = locations.get_by_id(target_loc)

if record:
    print(f"Target: {record['canonical_name']}")
    temp = record.get("temporal_validity", {})
    print(f"  Established: {temp.get('date_established')}")
    print(f"  Dissolved:   {temp.get('date_dissolved')}")
    
    for test_date in ["1820-05-15", "1895-10-10"]:
        valid = locations.is_valid_at_date(target_loc, test_date)
        print(f"  Extant at {test_date}? {valid}")
        
    successor = locations.get_successor(target_loc)
    if successor:
        print(f"  Successor Jurisdiction: [{successor['location_id']}] {successor['canonical_name']}")

In [ ]:
# 6. Adding Aliases & Safe Persistence to gtemp Scratch Target
# Append an alias to an existing jurisdiction
target_id = "LOC-00006"
added = locations.add_alias(target_id, "Carlisle Historic District")
print(f"Alias added: {added}")
print(f"Registry is dirty: {locations.is_dirty}")

# Persist exclusively to gtemp to protect canonical files
scratch_target = Path("gtemp/locations_scratch_output.json")
locations.save(output_path=scratch_target, force=True)
print(f"Safely written to: {scratch_target.resolve()}")